Installing Bayesian optimization package


In [1]:
pip install bayesian-optimization

Importing necessary libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from bayes_opt import BayesianOptimization
from bayes_opt import UtilityFunction
import matplotlib.pyplot as plt
from matplotlib import gridspec
%matplotlib inline

Creating a function g of variable x

In [3]:
import numpy as np
def Function(x):
    X=x
    g=X**2
    return g
x = np.linspace(-2, 10, 10000).reshape(-1, 1)
y=Function(x)

In [4]:
optimizer = BayesianOptimization(Function,{'x': (-2,10)}, random_state=27) #the region between x=-2 to 10 will be explicitely searched

In [5]:
acq_function = UtilityFunction(kind="ucb", kappa=10) #Upper Confidence Bound as acquisition function

In [8]:
#Defining the posterior
def posterior(optimizer, x_obs, y_obs, grid):
    optimizer._gp.fit(x_obs, y_obs)

    mu, sigma = optimizer._gp.predict(grid, return_std=True)
    return mu, sigma

def plot_gp(optimizer, x, y):
    fig = plt.figure(figsize=(16, 10))
    fig.dpi=300
    steps = len(optimizer.space)
    gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1])
    axis = plt.subplot(gs[0])
    acq = plt.subplot(gs[1])

    x_obs = np.array([[res["params"]["x"]] for res in optimizer.res])
    y_obs = np.array([res["target"] for res in optimizer.res])

    mu, sigma = posterior(optimizer, x_obs, y_obs, x)
    axis.plot(x, y, linewidth=3, label='Target')
    axis.plot(x_obs.flatten(), y_obs, 'D', markersize=8, label=u'Observations', color='r')
    axis.plot(x, mu, '--', color='k', label='Prediction')

    axis.fill(np.concatenate([x, x[::-1]]),
              np.concatenate([mu - 1.9600 * sigma, (mu + 1.9600 * sigma)[::-1]]),
        alpha=.6, fc='c', ec='None', label='95% confidence interval')
    axis.set_yticks([-20,0,20,40,60,80,100])
    plt.setp(axis.get_yticklabels(), weight='bold')
    plt.setp(axis.get_xticklabels(), weight='bold')

    axis.set_xlim((-2, 10))
    axis.set_ylim((None, None))
    axis.set_ylabel('f(x)', fontdict={'size':10},weight="bold")
    axis.set_xlabel('x', fontdict={'size':10},weight="bold")

    utility_function = UtilityFunction(kind="ucb", kappa=5, xi=0)
    utility = utility_function.utility(x, optimizer._gp, 0)
    acq.plot(x, utility, label='Utility Function', color='purple')
    acq.plot(x[np.argmax(utility)], np.max(utility), '*', markersize=15,
             label=u'Next Best Guess', markerfacecolor='gold', markeredgecolor='k', markeredgewidth=1)
    acq.set_xlim((-2, 10))
    acq.set_ylim((0, np.max(utility) + 0.5))
    acq.set_ylabel('Acquisition function', fontdict={'size':10},weight="bold")
    acq.set_xlabel('x', fontdict={'size':10},weight="bold")
    legend_properties = {'weight':'bold','size':'12'}
    axis.legend(prop=legend_properties)
    acq.legend(prop=legend_properties)
    acq.set_yticks([0,50,100])
    plt.setp(acq.get_yticklabels(), weight='bold')
    plt.setp(acq.get_xticklabels(), weight='bold')

    #axis.legend(loc=2, bbox_to_anchor=(1.01, 1), borderaxespad=0.)
    #acq.legend(loc=2, bbox_to_anchor=(1.01, 1), borderaxespad=0.)

In [ ]:
optimizer.maximize(init_points=2, n_iter=1, acquisition_function = acq_function) #two prior known points
plot_gp(optimizer, x, y)

In [ ]:
optimizer.maximize(init_points=0, n_iter=1)
plot_gp(optimizer, x, y)

In [ ]:
optimizer.maximize(init_points=0, n_iter=1, acquisition_function=acq_function)
plot_gp(optimizer, x, y)
plt.savefig('Figure1b.png', dpi=250)